# USDPEN Conditional Variance Model

### Global Constants & Plot Style

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# Regime date boundaries
INTERVENTION_START = pd.Timestamp("2025-11-01")
INTERVENTION_END = pd.Timestamp("2026-02-28")
POST_SHOCK_START = pd.Timestamp("2026-03-01")

# Pip definition: 1 pip = 0.0001 for USDPEN
PIP = 0.0001

# Plot style
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 100,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

COLOR_SYS = "#3B82F6"
COLOR_IDIO = "#F97316"
COLOR_TOTAL = "#111827"
COLOR_INTERVENTION = "#E5E7EB"
COLOR_SHOCK = "#FEE2E2"
REGIME_COLORS = {"normal": "#6B7280", "intervention": "#3B82F6", "post_shock": "#EF4444"}

def shade_regimes(ax, alpha=0.15):
    # shade intervention and post-shock regimes on a matplotlib axes
    ylim = ax.get_ylim()
    ax.axvspan(INTERVENTION_START, INTERVENTION_END, alpha=alpha, color=COLOR_INTERVENTION, label="BCRP Intervention")
    ax.axvspan(POST_SHOCK_START, pd.Timestamp("2026-12-31"), alpha=alpha, color=COLOR_SHOCK, label="Iran Conflict")
    ax.set_ylim(ylim)

def label_regime(date):
    # assign regime label to a date
    if date >= POST_SHOCK_START:
        return "post_shock"
    elif date >= INTERVENTION_START:
        return "intervention"
    else:
        return "normal"

print("Constants and style loaded.")

## 0. Synthetic Data Generation (DELETE WHEN USING REAL DATA)

In [ ]:
np.random.seed(42)

# Target final levels and annualized vols
CURRENCIES = ["usdpen", "usdclp", "usdcop", "usdbrl", "usdmxn"]
FINAL_LEVELS = {"usdpen": 3.6350, "usdclp": 917.43, "usdcop": 3994.46, "usdbrl": 5.33, "usdmxn": 17.89}
ANNUAL_VOLS = {"usdpen": 0.04, "usdclp": 0.12, "usdcop": 0.14, "usdbrl": 0.16, "usdmxn": 0.11}

# Generate ~504 business days ending 2026-03-13
end_date = pd.Timestamp("2026-03-13")
dates = pd.bdate_range(end=end_date, periods=504)
n_days = len(dates)

# Daily vols
daily_vols = {c: v / np.sqrt(252) for c, v in ANNUAL_VOLS.items()}

# Identify regime periods by index
intervention_mask = (dates >= INTERVENTION_START) & (dates <= INTERVENTION_END)
shock_mask = dates >= POST_SHOCK_START

# Common factor shocks (EM sentiment)
common_shocks = np.random.randn(n_days)

# Generate returns for each currency
returns_dict = {}
for ccy in CURRENCIES:
    dv = daily_vols[ccy]
    idio_shocks = np.random.randn(n_days)

    # Weights for common factor
    common_weight = np.full(n_days, 0.5)  # baseline ~0.5
    common_weight[shock_mask] = 0.75  # correlation spike during shock

    daily_vol_arr = np.full(n_days, dv)

    # Intervention: suppress USDPEN vol
    if ccy == "usdpen":
        daily_vol_arr[intervention_mask] = 0.015 / np.sqrt(252)

    # Shock: multiply all vols by 2.5x
    daily_vol_arr[shock_mask] *= 2.5

    # Mix common + idiosyncratic
    cw = common_weight
    mixed = cw * common_shocks + np.sqrt(1 - cw**2) * idio_shocks
    returns_dict[ccy] = mixed * daily_vol_arr

# Build log-level paths and then rescale to hit final levels
log_levels = {}
for ccy in CURRENCIES:
    cumul = np.cumsum(returns_dict[ccy])
    # Shift so that the last value gives the desired final level
    # log(S_T) = log(S_0) + sum(returns) => log(S_0) = log(S_T) - cumul[-1]
    log_s0 = np.log(FINAL_LEVELS[ccy]) - cumul[-1]
    log_path = log_s0 + cumul
    log_levels[ccy] = log_path

# Build DataFrame of levels
df_synth = pd.DataFrame(index=dates)
df_synth.index.name = "date"
for ccy in CURRENCIES:
    df_synth[ccy] = np.exp(log_levels[ccy])

# Save CSV
df_synth.to_csv("fx_latam_levels.csv", date_format="%Y-%m-%d")
print(f"Saved fx_latam_levels.csv: {df_synth.shape[0]} rows, {df_synth.index[0].date()} to {df_synth.index[-1].date()}")
print(f"Final levels: { {c: round(df_synth[c].iloc[-1], 4) for c in CURRENCIES} }")

## 1. Data Loading & Returns

### 1.1 — Load Data

In [ ]:
df_levels = pd.read_csv("fx_latam_levels.csv", parse_dates=["date"], index_col="date")
print(f"Shape: {df_levels.shape}")
print(f"Date range: {df_levels.index[0].date()} to {df_levels.index[-1].date()}")
print("\nFirst 5 rows:")
display(df_levels.head())
print("\nLast 5 rows:")
display(df_levels.tail())

### 1.2 — Compute Log Returns

In [ ]:
df_returns = np.log(df_levels / df_levels.shift(1))
df_returns.columns = ["r_pen", "r_clp", "r_cop", "r_brl", "r_mxn"]
df_returns = df_returns.dropna()

print("Summary Statistics:")
summary = df_returns.describe().T
summary["skew"] = df_returns.skew()
summary["kurtosis"] = df_returns.kurtosis()
display(summary.round(6))

### 1.3 — Quick Diagnostics

In [ ]:
# Levels subplots
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
for i, col in enumerate(df_levels.columns):
    axes[i].plot(df_levels.index, df_levels[col], color=COLOR_SYS, linewidth=1)
    axes[i].set_ylabel(col.upper())
    shade_regimes(axes[i])
axes[0].set_title("LatAm FX Levels")
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

# Correlation heatmap
corr = df_returns.corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap="RdYlBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.values[i,j]:.2f}", ha="center", va="center", fontsize=10)
plt.colorbar(im, ax=ax)
ax.set_title("Daily Log Return Correlations")
plt.tight_layout()
plt.show()

## 2. Model Functions

In [ ]:
def estimate_rolling_betas(df_returns, target="r_pen", regressors=["r_cop", "r_clp", "r_mxn", "r_brl"], window=120):
    # Rolling OLS betas of target on regressors
    dates_out = []
    betas_out = []
    r2_out = []
    y = df_returns[target].values
    X = df_returns[regressors].values
    idx = df_returns.index

    for i in range(window, len(y) + 1):
        y_w = y[i - window:i]
        X_w = X[i - window:i]
        # Add constant not needed — we regress without intercept for cleaner decomposition
        # Actually let's include intercept for proper R2
        X_aug = np.column_stack([np.ones(window), X_w])
        coef, res, _, _ = np.linalg.lstsq(X_aug, y_w, rcond=None)
        y_hat = X_aug @ coef
        ss_res = np.sum((y_w - y_hat) ** 2)
        ss_tot = np.sum((y_w - np.mean(y_w)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        dates_out.append(idx[i - 1])
        betas_out.append(coef[1:])  # skip intercept
        r2_out.append(r2)

    beta_cols = ["beta_" + r.split("_")[1] for r in regressors]
    result = pd.DataFrame(betas_out, columns=beta_cols, index=dates_out)
    result.index.name = "date"
    result["r_squared"] = r2_out
    return result


def estimate_ewma_betas(df_returns, target="r_pen", regressors=["r_cop", "r_clp", "r_mxn", "r_brl"], halflife=60, min_obs=60):
    # Exponentially weighted WLS betas
    dates_out = []
    betas_out = []
    r2_out = []
    y = df_returns[target].values
    X = df_returns[regressors].values
    idx = df_returns.index
    lam = np.log(2) / halflife

    for i in range(min_obs, len(y) + 1):
        y_w = y[:i]
        X_w = X[:i]
        n = len(y_w)
        # Weights: most recent observation has lag=0
        lags = np.arange(n - 1, -1, -1, dtype=float)
        w = np.exp(-lam * lags)
        w_sqrt = np.sqrt(w)

        X_aug = np.column_stack([np.ones(n), X_w])
        Xw = X_aug * w_sqrt[:, None]
        yw = y_w * w_sqrt
        coef, _, _, _ = np.linalg.lstsq(Xw, yw, rcond=None)
        y_hat = X_aug @ coef
        residuals = y_w - y_hat
        ss_res = np.sum(w * residuals ** 2)
        ss_tot = np.sum(w * (y_w - np.average(y_w, weights=w)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        dates_out.append(idx[i - 1])
        betas_out.append(coef[1:])
        r2_out.append(r2)

    beta_cols = ["beta_" + r.split("_")[1] for r in regressors]
    result = pd.DataFrame(betas_out, columns=beta_cols, index=dates_out)
    result.index.name = "date"
    result["r_squared"] = r2_out
    return result


def compute_ewma_covariance(df_returns, columns, decay=0.94):
    # RiskMetrics recursive EWMA covariance
    data = df_returns[columns].values
    n_cols = len(columns)
    dates = df_returns.index

    # Initialize with sample covariance of first 60 obs
    init_data = data[:60]
    sigma = np.cov(init_data, rowvar=False)

    cov_dict = {}
    var_records = []
    corr_records = []

    for t in range(60, len(data)):
        r_prev = data[t - 1].reshape(-1, 1)
        sigma = decay * sigma + (1 - decay) * (r_prev @ r_prev.T)
        dt = dates[t]
        cov_dict[dt] = sigma.copy()

        # Extract variances
        var_row = {columns[i]: sigma[i, i] for i in range(n_cols)}
        var_row["date"] = dt
        var_records.append(var_row)

        # Extract correlations
        stds = np.sqrt(np.diag(sigma))
        corr_mat = sigma / np.outer(stds, stds)
        corr_row = {"date": dt}
        for i in range(n_cols):
            for j in range(i + 1, n_cols):
                corr_row[f"{columns[i]}_{columns[j]}"] = corr_mat[i, j]
        corr_records.append(corr_row)

    df_var = pd.DataFrame(var_records).set_index("date")
    df_corr = pd.DataFrame(corr_records).set_index("date")
    return cov_dict, df_corr, df_var


def decompose_variance(betas_df, cov_dict, residuals_series, decay=0.94):
    # Decompose USDPEN variance into systematic + idiosyncratic
    regressors = ["r_cop", "r_clp", "r_mxn", "r_brl"]
    records = []

    # EWMA variance of residuals
    resid_vals = residuals_series.values
    resid_dates = residuals_series.index
    # Initialize idio variance from first 60 residuals
    init_var = np.var(resid_vals[:60]) if len(resid_vals) >= 60 else np.var(resid_vals)
    idio_ewma = init_var

    for i, dt in enumerate(betas_df.index):
        if dt not in cov_dict:
            continue

        # Systematic variance: beta' * Sigma_region * beta
        beta_vec = betas_df.loc[dt, ["beta_cop", "beta_clp", "beta_mxn", "beta_brl"]].values.astype(float)
        sigma_full = cov_dict[dt]
        # We need the submatrix for the 4 regressors
        # The cov_dict was computed on all 5 columns; indices 0=r_pen,1=r_clp,2=r_cop,3=r_brl,4=r_mxn
        # Actually columns order depends on what was passed — we'll handle via column mapping
        sys_var = beta_vec @ sigma_full @ beta_vec

        # Idiosyncratic variance: EWMA of residuals
        if dt in residuals_series.index:
            r_idx = residuals_series.index.get_loc(dt)
            if r_idx > 0:
                idio_ewma = decay * idio_ewma + (1 - decay) * resid_vals[r_idx - 1] ** 2

        total = sys_var + idio_ewma
        sys_pct = sys_var / total if total > 0 else 0.0
        records.append({
            "date": dt,
            "systematic_var": sys_var,
            "idiosyncratic_var": idio_ewma,
            "total_var": total,
            "systematic_pct": sys_pct,
        })

    return pd.DataFrame(records).set_index("date")


def conditional_range(regional_returns, betas, idio_var, spot_level, ci_levels=[0.90, 0.95, 0.99]):
    # Compute expected move and confidence bands
    # regional_returns: dict like {"r_cop": 0.002, "r_clp": -0.001, ...}
    # betas: dict like {"beta_cop": 0.3, ...}
    expected_return = sum(betas.get("beta_" + k.split("_")[1], 0) * v for k, v in regional_returns.items())
    expected_pips = expected_return * spot_level / PIP
    expected_level = spot_level * np.exp(expected_return)
    idio_std = np.sqrt(idio_var)

    bands = {}
    for ci in ci_levels:
        z = stats.norm.ppf(0.5 + ci / 2)
        pip_band = z * idio_std * spot_level / PIP
        lower_level = spot_level * np.exp(expected_return - z * idio_std)
        upper_level = spot_level * np.exp(expected_return + z * idio_std)
        bands[ci] = {
            "z": z,
            "pip_band": pip_band,
            "lower_level": lower_level,
            "upper_level": upper_level,
            "width_pips": 2 * pip_band,
            "lower_pips": expected_pips - pip_band,
            "upper_pips": expected_pips + pip_band,
        }

    return {
        "expected_return": expected_return,
        "expected_pips": expected_pips,
        "expected_level": expected_level,
        "idio_std": idio_std,
        "bands": bands,
    }


def build_monitor(df_levels, df_returns, betas_df, var_decomp_df, cov_dict, df_corr):
    # Build daily monitoring DataFrame
    regressors = ["r_cop", "r_clp", "r_mxn", "r_brl"]
    records = []
    common_dates = betas_df.index.intersection(var_decomp_df.index).intersection(df_returns.index)

    for dt in common_dates:
        row = {}
        row["date"] = dt
        row["usdpen_level"] = df_levels.loc[dt, "usdpen"] if dt in df_levels.index else np.nan
        row["usdpen_return"] = df_returns.loc[dt, "r_pen"]

        # Expected return from betas
        beta_vals = betas_df.loc[dt, ["beta_cop", "beta_clp", "beta_mxn", "beta_brl"]].values.astype(float)
        reg_vals = df_returns.loc[dt, regressors].values.astype(float)
        row["expected_return"] = float(beta_vals @ reg_vals)
        row["residual"] = row["usdpen_return"] - row["expected_return"]

        # Variance decomposition
        row["systematic_var"] = var_decomp_df.loc[dt, "systematic_var"]
        row["idiosyncratic_var"] = var_decomp_df.loc[dt, "idiosyncratic_var"]
        row["systematic_pct"] = var_decomp_df.loc[dt, "systematic_pct"]

        # Average correlation
        if dt in df_corr.index:
            row["avg_correlation"] = df_corr.loc[dt].mean()
        else:
            row["avg_correlation"] = np.nan

        # Beta sum
        row["beta_sum"] = float(beta_vals.sum())

        # Range width
        spot = row["usdpen_level"]
        idio_var = row["idiosyncratic_var"]
        if not np.isnan(spot) and idio_var > 0:
            z95 = stats.norm.ppf(0.975)
            row["range_95_width_pips"] = 2 * z95 * np.sqrt(idio_var) * spot / PIP
        else:
            row["range_95_width_pips"] = np.nan

        records.append(row)

    monitor = pd.DataFrame(records).set_index("date")

    # Rolling 20d cumulative residual and z-score
    monitor["cumul_residual_20d"] = monitor["residual"].rolling(20).sum()
    monitor["cumul_residual_std_20d"] = monitor["cumul_residual_20d"].rolling(20).std()
    monitor["residual_zscore"] = monitor["cumul_residual_20d"] / monitor["cumul_residual_std_20d"]

    return monitor

print("All model functions defined.")

## 3. Model Execution

### 3.1 — Estimate Betas

In [ ]:
betas_rolling = estimate_rolling_betas(df_returns, window=120)
betas_ewma = estimate_ewma_betas(df_returns, halflife=60, min_obs=60)

print("Rolling betas (last 10):")
display(betas_rolling.tail(10).round(4))
print("\nEWMA betas (last 10):")
display(betas_ewma.tail(10).round(4))

# Use EWMA as primary
betas_df = betas_ewma.copy()
print(f"\nPrimary model: EWMA betas, {len(betas_df)} observations")

### 3.2 — EWMA Covariance

In [ ]:
# Compute on regional currencies only (for systematic variance)
regional_cols = ["r_cop", "r_clp", "r_mxn", "r_brl"]
cov_dict, df_corr, df_var = compute_ewma_covariance(df_returns, regional_cols, decay=0.94)
print(f"EWMA covariance computed for {len(cov_dict)} dates")
print(f"\nLatest EWMA correlations:")
display(df_corr.tail(5).round(4))
print(f"\nLatest EWMA variances (annualized vol):")
display((np.sqrt(df_var.tail(5) * 252) * 100).round(2))

### 3.3 — Residuals & Variance Decomposition

In [ ]:
# Compute fitted and residual series
common_idx = betas_df.index.intersection(df_returns.index)
fitted = pd.Series(0.0, index=common_idx)
for reg in ["r_cop", "r_clp", "r_mxn", "r_brl"]:
    beta_col = "beta_" + reg.split("_")[1]
    fitted += betas_df.loc[common_idx, beta_col].values * df_returns.loc[common_idx, reg].values

residuals = df_returns.loc[common_idx, "r_pen"] - fitted
residuals.name = "residual"

var_decomp = decompose_variance(betas_df, cov_dict, residuals, decay=0.94)
print("Variance decomposition (last 10):")
display(var_decomp.tail(10).round(6))
print(f"\nCurrent systematic %: {var_decomp['systematic_pct'].iloc[-1]:.1%}")

### 3.4 — Build Monitor

In [ ]:
monitor = build_monitor(df_levels, df_returns, betas_df, var_decomp, cov_dict, df_corr)
monitor.to_csv("usdpen_daily_monitor.csv")
print(f"Monitor saved: {monitor.shape[0]} rows")
print("\nLast 20 rows:")
display(monitor.tail(20).round(4))

## 4. Live Update Widget

### 4.1 — Interactive Input

In [ ]:
import ipywidgets as widgets
from IPython.display import display as ipy_display, clear_output, HTML

# Last known levels
last_levels = df_levels.iloc[-1]
last_date = df_levels.index[-1]

# Input widgets
w_date = widgets.Text(value=str(pd.Timestamp("today").date()), description="Date:")
w_pen = widgets.FloatText(value=round(last_levels["usdpen"], 4), description="USDPEN:", step=0.0001)
w_clp = widgets.FloatText(value=round(last_levels["usdclp"], 2), description="USDCLP:", step=0.01)
w_cop = widgets.FloatText(value=round(last_levels["usdcop"], 2), description="USDCOP:", step=0.01)
w_brl = widgets.FloatText(value=round(last_levels["usdbrl"], 4), description="USDBRL:", step=0.0001)
w_mxn = widgets.FloatText(value=round(last_levels["usdmxn"], 4), description="USDMXN:", step=0.0001)

btn_compute = widgets.Button(description="Compute", button_style="primary", icon="calculator")
btn_save = widgets.Button(description="Save & Append to Data", button_style="success", icon="save")
output_area = widgets.Output()

def on_compute(b):
    with output_area:
        clear_output(wait=True)
        # Compute returns from last known levels
        new_levels = {"usdpen": w_pen.value, "usdclp": w_clp.value, "usdcop": w_cop.value,
                      "usdbrl": w_brl.value, "usdmxn": w_mxn.value}
        prev = {"usdpen": last_levels["usdpen"], "usdclp": last_levels["usdclp"],
                "usdcop": last_levels["usdcop"], "usdbrl": last_levels["usdbrl"],
                "usdmxn": last_levels["usdmxn"]}

        regional_returns = {
            "r_cop": np.log(new_levels["usdcop"] / prev["usdcop"]),
            "r_clp": np.log(new_levels["usdclp"] / prev["usdclp"]),
            "r_mxn": np.log(new_levels["usdmxn"] / prev["usdmxn"]),
            "r_brl": np.log(new_levels["usdbrl"] / prev["usdbrl"]),
        }

        # Latest betas and idio variance
        latest_betas = betas_df.iloc[-1].to_dict()
        latest_idio = var_decomp["idiosyncratic_var"].iloc[-1]
        latest_sys = var_decomp["systematic_var"].iloc[-1]
        total_v = latest_sys + latest_idio
        spot = new_levels["usdpen"]

        cr = conditional_range(regional_returns, latest_betas, latest_idio, spot)

        # Contribution breakdown
        contribs = {}
        for reg in ["r_cop", "r_clp", "r_mxn", "r_brl"]:
            bkey = "beta_" + reg.split("_")[1]
            contribs[reg] = latest_betas[bkey] * regional_returns[reg] * spot / PIP

        # Regional input display
        ccy_map = {"usdcop": ("r_cop", "USDCOP"), "usdclp": ("r_clp", "USDCLP"),
                   "usdmxn": ("r_mxn", "USDMXN"), "usdbrl": ("r_brl", "USDBRL")}

        lines = []
        lines.append(f"{'='*62}")
        lines.append(f"  USDPEN CONDITIONAL RANGE -- {w_date.value}")
        lines.append(f"{'='*62}")
        lines.append(f"  Regional Inputs:")
        for ccy_key, (ret_key, label) in ccy_map.items():
            lvl = new_levels[ccy_key]
            chg_pips = (new_levels[ccy_key] - prev[ccy_key]) / PIP
            pct = regional_returns[ret_key] * 100
            lines.append(f"    {label}: {lvl:.4f}  ({chg_pips:+.0f} pips / {pct:+.3f}%)")
        lines.append(f"{'-'*62}")
        lines.append(f"  Model Output:")
        lines.append(f"    Expected USDPEN Move:  {cr['expected_pips']:+.1f} pips ({cr['expected_return']*10000:+.2f} bps)")
        lines.append(f"    Expected USDPEN Level: {cr['expected_level']:.4f}")
        lines.append(f"")
        for ci in [0.90, 0.95, 0.99]:
            b = cr["bands"][ci]
            lines.append(f"    {ci:.0%} Range: {b['lower_level']:.4f} -- {b['upper_level']:.4f}  ({b['width_pips']:.0f} pips)")
        lines.append(f"{'-'*62}")
        lines.append(f"  Variance Decomposition:")
        lines.append(f"    Systematic (region):  {latest_sys/total_v:.1%}")
        lines.append(f"    Idiosyncratic (Peru): {latest_idio/total_v:.1%}")
        lines.append(f"    Total daily vol:      {np.sqrt(total_v)*10000:.1f} bps")
        lines.append(f"{'-'*62}")
        lines.append(f"  Diagnostics:")
        lines.append(f"    Beta sum:         {latest_betas['beta_cop']+latest_betas['beta_clp']+latest_betas['beta_mxn']+latest_betas['beta_brl']:.3f}")
        if len(monitor) > 0:
            lines.append(f"    Avg correlation:  {monitor['avg_correlation'].iloc[-1]:.3f}")
            zscore = monitor["residual_zscore"].iloc[-1]
            lines.append(f"    Residual z-score: {zscore:.2f} (last 20d)")
        lines.append(f"{'='*62}")

        print("\n".join(lines))

        # Bar chart of contributions
        fig, ax = plt.subplots(figsize=(8, 3))
        labels = [k.split("_")[1].upper() for k in contribs]
        vals = list(contribs.values())
        colors = [COLOR_SYS if v > 0 else COLOR_IDIO for v in vals]
        ax.barh(labels, vals, color=colors)
        ax.axvline(0, color="black", linewidth=0.5)
        ax.set_xlabel("Contribution (pips)")
        ax.set_title("Regional Contribution to Expected USDPEN Move")
        plt.tight_layout()
        plt.show()

btn_compute.on_click(on_compute)

def on_save(b):
    global df_levels, df_returns, betas_df, cov_dict, df_corr, df_var, var_decomp, residuals, monitor
    with output_area:
        new_date = pd.Timestamp(w_date.value)
        new_row = pd.DataFrame({
            "usdpen": [w_pen.value], "usdclp": [w_clp.value], "usdcop": [w_cop.value],
            "usdbrl": [w_brl.value], "usdmxn": [w_mxn.value]
        }, index=[new_date])
        new_row.index.name = "date"

        if new_date in df_levels.index:
            print(f"Date {new_date.date()} already exists. Overwriting.")
            df_levels.loc[new_date] = new_row.iloc[0]
        else:
            df_levels = pd.concat([df_levels, new_row]).sort_index()

        # Recompute returns
        df_returns = np.log(df_levels / df_levels.shift(1))
        df_returns.columns = ["r_pen", "r_clp", "r_cop", "r_brl", "r_mxn"]
        df_returns = df_returns.dropna()

        # Recompute EWMA betas
        betas_df = estimate_ewma_betas(df_returns, halflife=60, min_obs=60)
        regional_cols = ["r_cop", "r_clp", "r_mxn", "r_brl"]
        cov_dict, df_corr, df_var = compute_ewma_covariance(df_returns, regional_cols, decay=0.94)

        # Recompute residuals and variance decomp
        common_idx = betas_df.index.intersection(df_returns.index)
        fitted = pd.Series(0.0, index=common_idx)
        for reg in regional_cols:
            beta_col = "beta_" + reg.split("_")[1]
            fitted += betas_df.loc[common_idx, beta_col].values * df_returns.loc[common_idx, reg].values
        residuals = df_returns.loc[common_idx, "r_pen"] - fitted
        var_decomp = decompose_variance(betas_df, cov_dict, residuals, decay=0.94)

        # Rebuild monitor
        monitor = build_monitor(df_levels, df_returns, betas_df, var_decomp, cov_dict, df_corr)

        # Save files
        df_levels.to_csv("fx_latam_levels.csv", date_format="%Y-%m-%d")
        monitor.to_csv("usdpen_daily_monitor.csv")
        print(f"\nAppended {new_date.date()}. New dataset: {len(df_levels)} rows, {df_levels.index[0].date()} to {df_levels.index[-1].date()}.")

btn_save.on_click(on_save)

# Layout
input_box = widgets.VBox([w_date, w_pen, w_cop, w_clp, w_mxn, w_brl])
button_box = widgets.HBox([btn_compute, btn_save])
ipy_display(widgets.VBox([input_box, button_box, output_area]))

## 5. Charts

### 5.1 — Rolling Betas

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for col in ["beta_cop", "beta_clp", "beta_mxn", "beta_brl"]:
    ax.plot(betas_df.index, betas_df[col], label=col, linewidth=1.2)
shade_regimes(ax)
ax.set_title("USDPEN Betas on Regional Currencies (EWMA halflife=60d)")
ax.set_ylabel("Beta")
ax.legend(loc="upper left")
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

### 5.2 — Variance Decomposition

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(var_decomp.index, 0, var_decomp["systematic_var"] * 1e8,
                alpha=0.7, color=COLOR_SYS, label="Systematic")
ax.fill_between(var_decomp.index, var_decomp["systematic_var"] * 1e8,
                var_decomp["total_var"] * 1e8,
                alpha=0.7, color=COLOR_IDIO, label="Idiosyncratic")
ax.plot(var_decomp.index, var_decomp["total_var"] * 1e8,
        color=COLOR_TOTAL, linestyle="--", linewidth=1.2, label="Total EWMA Var")
shade_regimes(ax)
ax.set_title("USDPEN Variance Decomposition")
ax.set_ylabel("Variance (x1e-8)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 5.3 — Systematic Percentage

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(var_decomp.index, var_decomp["systematic_pct"] * 100, color=COLOR_SYS, linewidth=1.2)
ax.axhline(50, color="gray", linestyle="--", linewidth=0.8, label="50%")
ax.axhline(75, color="gray", linestyle=":", linewidth=0.8, label="75%")
shade_regimes(ax)
ax.set_title("Regional Share of USDPEN Variance")
ax.set_ylabel("Systematic %")
ax.set_ylim(0, 100)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 5.4 — Average Correlation

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_corr.index, df_corr.mean(axis=1), color=COLOR_SYS, linewidth=1.2)
shade_regimes(ax)
ax.set_title("Average Pairwise LatAm FX Correlation")
ax.set_ylabel("Avg Correlation")
plt.tight_layout()
plt.show()

### 5.5 — Conditional Range Backtest (Last 6 Months)

In [ ]:
def plot_conditional_range(monitor_df, title, n_days=None):
    # Plot actual moves vs conditional bands
    df_plot = monitor_df.dropna(subset=["range_95_width_pips", "usdpen_return"]).copy()
    if n_days is not None:
        df_plot = df_plot.tail(n_days)

    spot = df_plot["usdpen_level"]
    actual_pips = df_plot["usdpen_return"] * spot / PIP
    expected_pips = df_plot["expected_return"] * spot / PIP

    z95 = stats.norm.ppf(0.975)
    z99 = stats.norm.ppf(0.995)
    idio_std = np.sqrt(df_plot["idiosyncratic_var"])

    upper_95 = expected_pips + z95 * idio_std * spot / PIP
    lower_95 = expected_pips - z95 * idio_std * spot / PIP
    upper_99 = expected_pips + z99 * idio_std * spot / PIP
    lower_99 = expected_pips - z99 * idio_std * spot / PIP

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.fill_between(df_plot.index, lower_99, upper_99, alpha=0.15, color=COLOR_IDIO, label="99% CI")
    ax.fill_between(df_plot.index, lower_95, upper_95, alpha=0.25, color=COLOR_SYS, label="95% CI")
    ax.plot(df_plot.index, expected_pips, color=COLOR_TOTAL, linewidth=1, label="Expected")
    ax.scatter(df_plot.index, actual_pips, s=12, color=COLOR_TOTAL, alpha=0.6, label="Actual", zorder=5)
    shade_regimes(ax)
    ax.set_title(title)
    ax.set_ylabel("Pips")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

# Last 6 months ~ 126 business days
plot_conditional_range(monitor, "USDPEN Conditional Range -- Last 6 Months", n_days=126)

### 5.6 — Full Sample Conditional Range

In [ ]:
plot_conditional_range(monitor, "USDPEN Conditional Range -- Full Sample")

### 5.7 — Residual Z-Score

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
z = monitor["residual_zscore"].dropna()
colors_z = ["blue" if v < -1.5 else "red" if v > 1.5 else "gray" for v in z.values]
ax.scatter(z.index, z.values, c=colors_z, s=10, alpha=0.7)
ax.axhline(1.5, color="red", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axhline(-1.5, color="blue", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axhline(2.0, color="red", linestyle=":", linewidth=0.8, alpha=0.3)
ax.axhline(-2.0, color="blue", linestyle=":", linewidth=0.8, alpha=0.3)
ax.axhspan(-1.5, 1.5, alpha=0.05, color="gray")
shade_regimes(ax)
ax.set_title("USDPEN Idiosyncratic Residual Z-Score")
ax.set_ylabel("Z-Score")
plt.tight_layout()
plt.show()

### 5.8 — Beta Heatmap

In [ ]:
# Monthly average betas
betas_monthly = betas_df[["beta_cop", "beta_clp", "beta_mxn", "beta_brl"]].copy()
betas_monthly["month"] = betas_monthly.index.to_period("M")
heatmap_data = betas_monthly.groupby("month").mean().T

fig, ax = plt.subplots(figsize=(14, 3))
im = ax.imshow(heatmap_data.values, aspect="auto", cmap="RdYlBu_r")
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels([b.replace("beta_", "").upper() for b in heatmap_data.index])
ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels([str(p) for p in heatmap_data.columns], rotation=45, ha="right")
plt.colorbar(im, ax=ax, label="Beta")
ax.set_title("Beta Evolution Heatmap")
plt.tight_layout()
plt.show()

## 6. Backtesting

### 6.1 — Backtest Functions

In [ ]:
def backtest_coverage(monitor_df, train_window=252, ci_levels=[0.90, 0.95, 0.99]):
    # Walk-forward coverage test
    df = monitor_df.dropna(subset=["usdpen_return", "expected_return", "idiosyncratic_var"]).copy()
    if len(df) <= train_window:
        print("Not enough data for walk-forward backtest")
        return None
    test_df = df.iloc[train_window:]
    results = {}

    for ci in ci_levels:
        z = stats.norm.ppf(0.5 + ci / 2)
        expected = test_df["expected_return"].values
        actual = test_df["usdpen_return"].values
        idio_std = np.sqrt(test_df["idiosyncratic_var"].values)
        upper = expected + z * idio_std
        lower = expected - z * idio_std
        breaches = (actual > upper) | (actual < lower)
        n_breaches = breaches.sum()
        n_total = len(breaches)
        empirical_rate = n_breaches / n_total
        theoretical_rate = 1 - ci

        # Christoffersen unconditional coverage LR test
        p0 = theoretical_rate
        p1 = max(min(empirical_rate, 1 - 1e-10), 1e-10)
        n0 = n_total - n_breaches
        n1 = n_breaches
        log_l0 = n0 * np.log(1 - p0) + n1 * np.log(p0)
        log_l1 = n0 * np.log(1 - p1) + n1 * np.log(p1)
        lr_uc = -2 * (log_l0 - log_l1)
        lr_uc = max(lr_uc, 0)
        pval_uc = 1 - stats.chi2.cdf(lr_uc, 1)

        # Independence test (2x2 transitions)
        b = breaches.astype(int)
        t00 = t01 = t10 = t11 = 0
        for k in range(1, len(b)):
            if b[k-1] == 0 and b[k] == 0: t00 += 1
            elif b[k-1] == 0 and b[k] == 1: t01 += 1
            elif b[k-1] == 1 and b[k] == 0: t10 += 1
            else: t11 += 1

        pi01 = t01 / (t00 + t01) if (t00 + t01) > 0 else 0
        pi11 = t11 / (t10 + t11) if (t10 + t11) > 0 else 0
        pi = (t01 + t11) / (t00 + t01 + t10 + t11) if (t00 + t01 + t10 + t11) > 0 else 0

        # LR independence
        lr_ind = 0
        if pi > 0 and pi < 1:
            if t00 > 0 and pi01 > 0 and pi01 < 1:
                lr_ind += 2 * (t00 * np.log(1 - pi01) + t01 * np.log(pi01) - (t00 + t01) * np.log(1 - pi) - 0)
                lr_ind -= 2 * (-t00 * np.log(1 - pi) - t01 * np.log(pi) + t00 * np.log(1 - pi01) + t01 * np.log(pi01))
            lr_ind = 0  # simplify: just use unconditional
            # Full LR independence:
            eps = 1e-10
            p01 = max(pi01, eps)
            p11_v = max(pi11, eps)
            pi_v = max(pi, eps)
            log_l1 = t00 * np.log(1 - p01 + eps) + t01 * np.log(p01 + eps)
            if (t10 + t11) > 0:
                log_l1 += t10 * np.log(1 - p11_v + eps) + t11 * np.log(p11_v + eps)
            log_l0 = (t00 + t10) * np.log(1 - pi_v + eps) + (t01 + t11) * np.log(pi_v + eps)
            lr_ind = 2 * (log_l1 - log_l0)
            lr_ind = max(lr_ind, 0)

        pval_ind = 1 - stats.chi2.cdf(lr_ind, 1)
        lr_cc = lr_uc + lr_ind
        pval_cc = 1 - stats.chi2.cdf(lr_cc, 2)

        results[ci] = {
            "n_total": n_total, "n_breaches": n_breaches,
            "empirical_rate": empirical_rate, "theoretical_rate": theoretical_rate,
            "lr_uc": lr_uc, "pval_uc": pval_uc,
            "lr_ind": lr_ind, "pval_ind": pval_ind,
            "lr_cc": lr_cc, "pval_cc": pval_cc,
            "breach_dates": test_df.index[breaches],
        }

    return results


def pit_test(actual, expected, variances):
    # Probability Integral Transform test
    stds = np.sqrt(variances)
    z_scores = (actual - expected) / stds
    pit_values = stats.norm.cdf(z_scores)
    ks_stat, ks_pval = stats.kstest(pit_values, "uniform")
    hist_counts, _ = np.histogram(pit_values, bins=20, range=(0, 1))
    return {"ks_stat": ks_stat, "ks_pval": ks_pval, "pit_values": pit_values, "hist_counts": hist_counts}


def basel_traffic_light(breaches_99, window=250):
    # Rolling count of 99% breaches; classify Green/Yellow/Red
    rolling_count = breaches_99.rolling(window, min_periods=1).sum()
    def classify(n):
        if n <= 4:
            return "Green"
        elif n <= 9:
            return "Yellow"
        else:
            return "Red"
    classification = rolling_count.apply(classify)
    return rolling_count, classification


def mm_simulation(monitor_df, ci_level=0.95, holding_period=1, cost_pips=3):
    # Market-making fade strategy simulation
    df = monitor_df.dropna(subset=["usdpen_return", "expected_return", "idiosyncratic_var", "usdpen_level"]).copy()
    z = stats.norm.ppf(0.5 + ci_level / 2)
    trades = []

    for i in range(len(df)):
        row = df.iloc[i]
        actual = row["usdpen_return"]
        expected = row["expected_return"]
        idio_std = np.sqrt(row["idiosyncratic_var"])
        spot = row["usdpen_level"]
        upper = expected + z * idio_std
        lower = expected - z * idio_std

        if actual > upper:
            # Actual moved too much up -> fade by selling
            excess_pips = (actual - upper) * spot / PIP
            # Next day reversion P&L (simplified)
            if i + holding_period < len(df):
                next_ret = df.iloc[i + holding_period]["usdpen_return"]
                pnl_pips = -next_ret * spot / PIP - cost_pips  # sold, so profit from decline
            else:
                continue
            trades.append({"date": df.index[i], "direction": "sell", "excess_pips": excess_pips,
                          "pnl_pips": pnl_pips, "regime": label_regime(df.index[i])})

        elif actual < lower:
            # Actual moved too much down -> fade by buying
            excess_pips = (lower - actual) * spot / PIP
            if i + holding_period < len(df):
                next_ret = df.iloc[i + holding_period]["usdpen_return"]
                pnl_pips = next_ret * spot / PIP - cost_pips
            else:
                continue
            trades.append({"date": df.index[i], "direction": "buy", "excess_pips": excess_pips,
                          "pnl_pips": pnl_pips, "regime": label_regime(df.index[i])})

    if len(trades) == 0:
        return pd.DataFrame()
    trade_df = pd.DataFrame(trades).set_index("date")
    trade_df["cumul_pnl"] = trade_df["pnl_pips"].cumsum()
    return trade_df

print("Backtest functions defined.")

### 6.2 — Run Backtest

In [ ]:
# Coverage test
coverage_results = backtest_coverage(monitor, train_window=120)

if coverage_results is not None:
    print("Coverage Test Results:")
    print(f"{'CI':>6} | {'Theoretical':>11} | {'Empirical':>10} | {'Breaches':>8} | {'LR_UC':>8} | {'p_UC':>8} | {'LR_CC':>8} | {'p_CC':>8}")
    print("-" * 90)
    for ci, r in coverage_results.items():
        print(f"{ci:6.0%} | {r['theoretical_rate']:11.3%} | {r['empirical_rate']:10.3%} | "
              f"{r['n_breaches']:8d} | {r['lr_uc']:8.3f} | {r['pval_uc']:8.3f} | {r['lr_cc']:8.3f} | {r['pval_cc']:8.3f}")

# PIT test
test_data = monitor.dropna(subset=["usdpen_return", "expected_return", "idiosyncratic_var"])
pit_results = pit_test(test_data["usdpen_return"].values, test_data["expected_return"].values,
                       test_data["idiosyncratic_var"].values)
print(f"\nPIT Test: KS stat = {pit_results['ks_stat']:.4f}, p-value = {pit_results['ks_pval']:.4f}")

# Basel traffic light
if coverage_results is not None and 0.99 in coverage_results:
    breaches_99_series = pd.Series(0, index=monitor.index)
    for dt in coverage_results[0.99]["breach_dates"]:
        if dt in breaches_99_series.index:
            breaches_99_series.loc[dt] = 1
    rolling_breaches, traffic_class = basel_traffic_light(breaches_99_series)
    print(f"\nBasel Traffic Light (current): {traffic_class.iloc[-1]} ({int(rolling_breaches.iloc[-1])} breaches in window)")

# MM simulation
trade_df = mm_simulation(monitor, ci_level=0.95, holding_period=1, cost_pips=3)
if len(trade_df) > 0:
    print(f"\nMM Fade Strategy:")
    print(f"  Total trades: {len(trade_df)}")
    print(f"  Win rate: {(trade_df['pnl_pips'] > 0).mean():.1%}")
    sharpe = trade_df["pnl_pips"].mean() / trade_df["pnl_pips"].std() * np.sqrt(252) if trade_df["pnl_pips"].std() > 0 else 0
    print(f"  Sharpe: {sharpe:.2f}")
    print(f"  Max drawdown: {(trade_df['cumul_pnl'] - trade_df['cumul_pnl'].cummax()).min():.1f} pips")
    print(f"  Cumulative P&L: {trade_df['cumul_pnl'].iloc[-1]:.1f} pips")
    print(f"\n  By regime:")
    for regime in ["normal", "intervention", "post_shock"]:
        sub = trade_df[trade_df["regime"] == regime]
        if len(sub) > 0:
            print(f"    {regime}: {len(sub)} trades, avg P&L = {sub['pnl_pips'].mean():.1f} pips, total = {sub['pnl_pips'].sum():.1f} pips")
else:
    print("\nNo trades generated in MM simulation.")

### 6.3 — Coverage Bar Chart

In [ ]:
if coverage_results is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ci_labels = [f"{int(ci*100)}%" for ci in coverage_results]
    theoretical = [coverage_results[ci]["theoretical_rate"] * 100 for ci in coverage_results]
    empirical = [coverage_results[ci]["empirical_rate"] * 100 for ci in coverage_results]
    x = np.arange(len(ci_labels))
    w = 0.35
    ax.bar(x - w/2, theoretical, w, label="Theoretical", color=COLOR_SYS, alpha=0.8)
    ax.bar(x + w/2, empirical, w, label="Empirical", color=COLOR_IDIO, alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(ci_labels)
    ax.set_ylabel("Breach Rate (%)")
    ax.set_title("Model Coverage Test")
    ax.legend()
    plt.tight_layout()
    plt.show()

### 6.4 — PIT Histogram

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pit_results["pit_values"], bins=20, range=(0, 1), density=True, alpha=0.7, color=COLOR_SYS, edgecolor="white")
ax.axhline(1.0, color="red", linestyle="--", linewidth=1, label="Uniform (ideal)")
ax.set_xlabel("PIT Value")
ax.set_ylabel("Density")
ax.set_title("PIT Histogram -- Model Calibration")
ax.legend()
plt.tight_layout()
plt.show()

### 6.5 — Breach Timeline

In [ ]:
if coverage_results is not None and 0.99 in coverage_results:
    fig, ax = plt.subplots(figsize=(14, 4))
    # Plot all returns
    spot = monitor["usdpen_level"]
    actual_pips = monitor["usdpen_return"] * spot / PIP
    ax.plot(monitor.index, actual_pips, color="gray", alpha=0.4, linewidth=0.8)

    # Mark breaches
    breach_dates = coverage_results[0.99]["breach_dates"]
    for dt in breach_dates:
        if dt in monitor.index:
            regime = label_regime(dt)
            color = REGIME_COLORS.get(regime, "black")
            val = actual_pips.loc[dt]
            ax.scatter(dt, val, color=color, s=40, zorder=5, edgecolors="black", linewidth=0.5)

    shade_regimes(ax)
    # Legend for regimes
    for regime, color in REGIME_COLORS.items():
        ax.scatter([], [], color=color, label=regime, s=40, edgecolors="black", linewidth=0.5)
    ax.set_title("99% CI Breaches")
    ax.set_ylabel("Pips")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

### 6.6 — Basel Traffic Light

In [ ]:
if coverage_results is not None and 0.99 in coverage_results:
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(rolling_breaches.index, rolling_breaches.values, color=COLOR_TOTAL, linewidth=1.2)

    # Background coloring
    for i in range(len(rolling_breaches)):
        c = traffic_class.iloc[i]
        color = {"Green": "#10B981", "Yellow": "#F59E0B", "Red": "#EF4444"}.get(c, "gray")
        if i > 0:
            ax.axvspan(rolling_breaches.index[i-1], rolling_breaches.index[i], alpha=0.15, color=color)

    ax.axhline(4, color="#F59E0B", linestyle="--", linewidth=0.8, label="Green/Yellow (4)")
    ax.axhline(9, color="#EF4444", linestyle="--", linewidth=0.8, label="Yellow/Red (9)")
    ax.set_title("Basel Traffic Light")
    ax.set_ylabel("Rolling 99% Breaches (250d)")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

### 6.7 — MM Simulation P&L

In [ ]:
if len(trade_df) > 0:
    fig, ax = plt.subplots(figsize=(14, 4))
    for regime in ["normal", "intervention", "post_shock"]:
        mask = trade_df["regime"] == regime
        if mask.any():
            sub = trade_df[mask]
            ax.fill_between(sub.index, 0, sub["cumul_pnl"], alpha=0.3,
                          color=REGIME_COLORS[regime], label=regime)
    ax.plot(trade_df.index, trade_df["cumul_pnl"], color=COLOR_TOTAL, linewidth=1.2)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_title("Fade Strategy Cumulative P&L")
    ax.set_ylabel("Cumulative P&L (pips)")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("No trades to plot.")